In [ ]:
# RNNによる文章生成
# 言語モデルを使った文章生成
# RNNによる文章生成の手順
# 文章生成の実装
import sys
sys.path.append('../..')
import numpy as np
from common.functions import softmax
from ch06.rnnlm import Rnnlm
from ch06.better_rnnlm import BetterRnnlm


class RnnlmGen(Rnnlm):
    # start_id：文章の最初の単語ID
    # skip_ids：生成をスキップする単語IDのリスト
    # sample_size：生成する単語の総数
    def generate(self, start_id, skip_ids=None, sample_size=100):
        # word_ids：生成された単語IDを順番に記録していくためのリスト
        # 最初はstart_idしか渡されないので、start_idをリストに追加
        word_ids = [start_id]

        # 次のRNNモデルに入力するための変数xを作成
        # 最初はstart_id ➡ ループ中で新しく生成された単語に上書きされていく
        x = start_id
        while len(word_ids) < sample_size:  # リストの長さ（生成された単語数）がsample_sizeの値になるまで繰り返す
            # 今持ってる入力単語をNumpy配列に変換し、1行1列に変形
            x = np.array(x).reshape(1, 1)
            # 第6章で作成したBetterRnnlmのpredict
            score = self.predict(x)
            # ソフトマックス関数を通して確率分布を求める
            # flatten()：1次元配列に変換（softmax関数は1次元配列の入力を前提として作られているため）
            p = softmax(score.flatten())

            # 確率分布pから1つの単語をサンプリング
            sampled = np.random.choice(len(p), size=1, p=p)
            # サンプリングした単語がskip_idsに含まれていない場合、
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled  # サンプリングした値を新しい入力単語へ
                word_ids.append(int(x))  # 生成された単語を追加

        return word_ids

    # 現在のLSTM層が憶えている隠れ状態hと記憶セルcを取得
    def get_state(self):
        return self.lstm_layer.h, self.lstm_layer.c

    # 外から保存された記憶（state）を受け取ってLSTMにセット
    def set_state(self, state):
        self.lstm_layer.set_state(*state)


In [8]:
# さらに良い文章へ

In [ ]:
# seq2seq
# seq2seqの原理
# 時系列データ変換用のトイ・プロブレム
# 可変長の時系列データ
# 足し算データセット
import sys
sys.path.append('..')
from dataset import sequence

# 訓練データとテストデータに分割
(x_train, t_train), (x_test, t_test) = \
    sequence.load_data('addition.txt', seed=1984)
char_to_id, id_to_char = sequence.get_vocab()

print(x_train.shape, t_train.shape)
print(x_test.shape, t_test.shape)
# (45000, 7) (45000, 5)
# (5000, 7) (5000, 5)

print(x_train[0])
print(t_train[0])
# [ 3  0  2  0  0 11  5]
# [ 6  0 11  7  5]

print(''.join([id_to_char[c] for c in x_train[0]]))
print(''.join([id_to_char[c] for c in t_train[0]]))
# 71+118
# _189

(45000, 7) (45000, 5)
(5000, 7) (5000, 5)
[ 3  0  2  0  0 11  5]
[ 6  0 11  7  5]
71+118 
_189 


In [10]:
# seq2seqの実装
# Encoderクラス
import sys
sys.path.append("..")
from common.time_layers import *

class Encoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)

        self.params = self.embed.params + self.lstm.params
        self.grads = self.embed.grads + self.lstm.grads
        self.hs = None

    def forward(self, xs):
        xs = self.embed.forward(xs)
        hs = self.lstm.forward(xs)
        self.hs = hs
        return hs[:, -1, :]

    def backward(self, dh):
        dhs = np.zeros_like(self.hs)
        dhs[:, -1, :] = dh

        dout = self.lstm.backward(dhs)
        dout = self.embed.backward(dout)
        return dout

In [ ]:
# Decoderクラス
class Decoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True)
        self.affine = TimeAffine(affine_W, affine_b)

        self.params, self.grads = [], []
        for layer in (self.embed, self.lstm, self.affine):
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, xs, h):
        self.lstm.set_state(h)

        out = self.embed.forward(xs)
        out = self.lstm.forward(out)
        score = self.affine.forward(out)
        return score

    def backward(self, dscore):
        dout = self.affine.backward(dscore)
        dout = self.lstm.backward(dout)
        dout = self.embed.backward(dout)
        dh = self.lstm.dh
        return dh

    def generate(self, h, start_id, sample_size):
        sampled = []    # モデルが生成した数字や文字を一文字ずつ入れるためのリスト
        sample_id = start_id  # 今から入力する最初のIDをセット
        self.lstm.set_state(h)

        for _ in range(sample_size):  # サンプルサイズの数だけループを回す
            # 入力IDをNumpy配列に変換し、1行1列に変更（入力がただの数字のため）
            x = np.array(sample_id).reshape((1, 1))
            out = self.embed.forward(x)
            out = self.lstm.forward(out)
            score = self.affine.forward(out)

            # リストの中で一番大きい値のインデックスを取得
            sample_id = np.argmax(score.flatten())
            sampled.append(int(sample_id))

        return sampled

In [ ]:
# Seq2seqクラス
from common.base_model import BaseModel

class Seq2seq(BaseModel):
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        self.encoder = Encoder(V, D, H)
        self.decoder = Decoder(V, D, H)
        self.softmax = TimeSoftmaxWithLoss()

        self.params = self.encoder.params + self.decoder.params
        self.grads = self.encoder.grads + self.decoder.grads

    def forward(self, xs, ts):
        decoder_xs, decoder_ts = ts[:, :-1], ts[:, 1:]

        h = self.encoder.forward(xs)
        score = self.decoder.forward(decoder_xs, h)
        loss = self.softmax.forward(score, decoder_ts)
        return loss

    def backward(self, dout=1):
        dout = self.softmax.backward(dout)
        dh = self.decoder.backward(dout)
        dout = self.encoder.backward(dh)
        return dout

    def generate(self, xs, start_id, sample_size):
        h = self.encoder.forward(xs)
        sampled = self.decoder.generate(h, start_id, sample_size)
        return sampled